# Project 8: Customer Segmentation — Clustering

**Dataset:** Mall Customers (200 customers, Income + Spending Score)

**Goal:** Discover natural customer segments using K-Means, Hierarchical clustering, and DBSCAN. Compare how each algorithm works.

In [ ]:
# ====== Setup: imports + data loader ======
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
import warnings; warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.dpi'] = 120

DATA_URL = 'https://raw.githubusercontent.com/lucyyuhongxia-gmail/A4_ML_Projects/main/datasets'
def load_data(fn):
    """Load a real dataset from GitHub."""
    df = pd.read_csv(f"{DATA_URL}/{fn}")
    print(f"Loaded: {fn} -> {df.shape[0]:,} rows x {df.shape[1]} cols")
    return df
print('Setup OK!')
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage

## 1. Load and Explore Customer Data

In [ ]:
# Mall Customers dataset: 200 customers
# Each customer has: Age, Annual Income, Spending Score (1-100)
df = load_data('mall_customers.csv')
print('First 5 rows:')
print(df.head().to_string())
print(f'\nShape: {df.shape}')

# Features for clustering — 用于聚类的特征 (收入+购物分)
X = df[['AnnualIncome_k', 'SpendingScore']]
X_scaled = StandardScaler().fit_transform(X)

# Quick EDA
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df['AnnualIncome_k'], bins=20, kde=True, ax=axes[0])
axes[0].set_title('Annual Income Distribution')
sns.histplot(df['SpendingScore'], bins=20, kde=True, ax=axes[1])
axes[1].set_title('Spending Score Distribution')
plt.tight_layout(); plt.show()

## 2. K-Means — Finding the Right K

K-Means is the most popular clustering algorithm. But we need to choose the number of clusters (K) in advance. We use two methods:

- **Elbow Method**: Plot inertia vs K. Choose K at the 'elbow'.
- **Silhouette Score**: Higher is better. Measures cluster separation.

In [ ]:
# Elbow Method + Silhouette Analysis — 肘部法 + 轮廓系数
inertias, sil_scores = [], []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))
    print(f'k={k:2d}: Inertia={km.inertia_:.0f}, Silhouette={sil_scores[-1]:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(K_range, inertias, 'o-', color='#2e86de', linewidth=2)
axes[0].set_xlabel('k (number of clusters)')
axes[0].set_ylabel('Inertia (lower = better)')
axes[0].set_title('Elbow Method'); axes[0].grid(True, alpha=0.3)

axes[1].plot(K_range, sil_scores, 'o-', color='#e74c3c', linewidth=2)
best_k = K_range[sil_scores.index(max(sil_scores))]
axes[1].axvline(x=best_k, color='green', ls=':', label=f'Best k={best_k}')
axes[1].set_xlabel('k (number of clusters)')
axes[1].set_ylabel('Silhouette Score (higher = better)')
axes[1].set_title('Silhouette Analysis'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 3. Visualize Customer Segments

In [ ]:
# Apply K-Means with the optimal k — 使用最佳k值聚类
k = 5
km = KMeans(n_clusters=k, random_state=42, n_init=10)
df['Cluster'] = km.fit_predict(X_scaled)

# Visualize
fig, ax = plt.subplots(figsize=(10, 8))
sc = ax.scatter(df['AnnualIncome_k'], df['SpendingScore'],
                c=df['Cluster'], cmap='viridis', s=60, alpha=0.7,
                edgecolors='black', linewidth=0.5)
ax.scatter(km.cluster_centers_[:,0]*X.std().AnnualIncome_k + X.mean().AnnualIncome_k,
           km.cluster_centers_[:,1]*X.std().SpendingScore + X.mean().SpendingScore,
           c='red', marker='X', s=200, edgecolors='black', linewidth=2,
           label='Centroids')
ax.set_xlabel('Annual Income ($K)'); ax.set_ylabel('Spending Score')
ax.set_title('K-Means Customer Segments (k=5)')
ax.legend(); ax.grid(True, alpha=0.3)
plt.colorbar(sc, ax=ax, label='Cluster')
plt.tight_layout(); plt.show()

# Profile each cluster — 分析每个分群的特征
print('\nCluster Profiles:')
for c in sorted(df.Cluster.unique()):
    sub = df[df.Cluster == c]
    print(f'  Cluster {c}: n={len(sub):2d}, Age={sub.Age.mean():.0f}, '
          f'Income=${sub.AnnualIncome_k.mean():.0f}K, '
          f'Spending={sub.SpendingScore.mean():.0f}')

## 4. Hierarchical & DBSCAN Clustering

Two alternative approaches:

- **Hierarchical**: Builds a tree of clusters (dendrogram). No need to choose K in advance.
- **DBSCAN**: Density-based. Finds arbitrary shapes and automatically detects noise points.

In [ ]:
# Hierarchical Clustering — 层次聚类树状图
fig, ax = plt.subplots(figsize=(12, 6))
Z = linkage(X_scaled, method='ward')
dendrogram(Z, truncate_mode='lastp', p=30, leaf_rotation=45)
ax.set_title('Dendrogram — Hierarchical Clustering')
ax.set_xlabel('Samples'); ax.set_ylabel('Distance')
plt.tight_layout(); plt.show()

# DBSCAN — 密度聚类 (自动检测噪声点)
db = DBSCAN(eps=0.5, min_samples=5)
labels = db.fit_predict(X_scaled)
n_clusters = len(set(labels) - {-1})
n_noise = sum(labels == -1)
print(f'DBSCAN (eps=0.5): {n_clusters} clusters, {n_noise} noise points ({n_noise/200:.1%})')

print('\nAlgorithm Comparison:')
print('  K-Means:      Fast, spherical, needs K')
print('  Hierarchical: Tree view, no K needed, expensive')
print('  DBSCAN:       Arbitrary shapes, noise detection')

## Check Your Understanding

1. **Elbow Method:** Looking at the inertia plot, what k would you choose? Justify your answer.

2. **Silhouette Score:** What range does the score take? Is k=5's score of ~0.44 good?

3. **Cluster Profiles:** Which cluster would you target with a VIP loyalty program? Which cluster needs gentle encouragement to spend more?

4. **DBSCAN:** Why did DBSCAN find only 1 main cluster (with a few noise points)? What does this tell you about the data's structure?

5. **Business Action:** As a mall manager, what would you do differently for each of the 5 segments?